# ShopSmart: Purchase Prediction Using Decision Trees

**Objective:** Predict whether a customer will generate revenue (make a purchase) based on their browsing behaviour and session data from an e-commerce platform.

**Approach:** Build a Decision Tree classifier and optimise it using pre-pruning and post-pruning techniques to improve generalisation.

**Dataset:** `shop_smart_ecommerce.csv`

**Metric Used:** F1-Score (chosen due to class imbalance between buyers and non-buyers)

## 1. Importing Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

## 2. Loading the Dataset

In [ ]:
df = pd.read_csv("shop_smart_ecommerce.csv")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

Inspect the column types, null values, and basic structure of the dataset.

In [ ]:
df.info() #Montha and visitor are in categories

In [ ]:
df.head()

### 3.2 Class Distribution — Revenue

Checking for class imbalance. An imbalanced target variable means accuracy alone is misleading — hence F1-Score is used.

In [ ]:
#How balanced our output classes?

class_cnt = df["Revenue"].value_counts()
plt.pie(class_cnt, labels=["No","Yes"], autopct="%1.1f%%")
plt.title("Is Revenue genereated?")

### 3.3 Visitor Type Distribution

Most users are returning visitors. This feature will be one-hot encoded before training.

In [ ]:
#checking distributions of all categorical features

class_cnt = df["VisitorType"].value_counts()
plt.pie(class_cnt,labels=["Returning_visitor","New_visitor","Others"], autopct="%1.1f%%")
plt.title("Visitor Type")
class_cnt

### 3.4 Monthly Revenue Distribution

Revenue activity peaks in November — likely driven by Black Friday / Diwali sales. No data exists for January and April.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Define the correct chronological order for 10 existing months
month_order = ["Feb","Mar","May","June","Jul","Aug","Sep","Oct","Nov","Dec"]

# 2. Setting the visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 5))

# 3. Bar chart with the forced 'order' parameter
sns.barplot(
    data=df, x="Month", y="Revenue", order=month_order, color="b"
)

# 4. Clean up the labels
plt.title("Customer Distribution by Month (Excluding Jan & Apr)", fontsize=14)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)

plt.show()


## 4. Data Preprocessing

### 4.1 Encoding Categorical Features

- `Revenue` and `Weekend` are label-encoded (binary).
- `VisitorType` and `Month` are one-hot encoded with `drop_first=True` to avoid multicollinearity.

In [ ]:
#Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Revenue"] = le.fit_transform(df["Revenue"])
df["Weekend"] = le.fit_transform(df["Weekend"])

df = pd.get_dummies(df,columns = ["VisitorType"],drop_first=True,dtype=int)
df = pd.get_dummies(df,columns = ["Month"],drop_first = True,dtype=int)

In [ ]:
df.info()

## 5. Baseline Decision Tree Model

### 5.1 Train-Test Split

Split data into 80% training and 20% testing. `random_state=42` ensures reproducibility.

In [ ]:
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,precision_score,accuracy_score,recall_score

X = df.drop("Revenue",axis=1)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y , test_size=0.2, random_state=42
)

### 5.2 Training the Baseline Model

Train an unpruned Decision Tree with default hyperparameters. This will likely overfit.

In [ ]:
model = DecisionTreeClassifier()
model.fit(X_train,y_train)

### 5.3 Baseline Model Evaluation

Evaluate on the test set. Low F1-Score expected due to overfitting on the training data.

In [ ]:
y_pred = model.predict(X_test)

print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy score: ",accuracy_score(y_test,y_pred))
print("Precision score: ",precision_score(y_test,y_pred))
print("Recall score: ",recall_score(y_test,y_pred))


### 5.4 Visualising the Baseline Tree

The unpruned tree is extremely deep — a sign of overfitting.

In [ ]:
plt.figure(figsize=(12,8))
plot_tree(
    model,
    feature_names = X.columns,
    class_names = ["No","Yes"],
    filled=True
)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning — Pre-Pruning

Pre-pruning restricts tree growth during training. We use `GridSearchCV` with 5-fold cross-validation to tune two key parameters:

- **`max_depth`** — limits how deep the tree grows
- **`min_samples_leaf`** — minimum samples required at a leaf node

A `StandardScaler` is added inside the pipeline to ensure fair comparisons across splits.

In [ ]:
#Pre-pruning
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline(
    [('Scaler',StandardScaler()),
    ('dt',DecisionTreeClassifier(class_weight="balanced"))])
param_grid = {"dt__max_depth":[2,3,4,5,6,7,8,9,10]}

classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)
classifierCV.fit(X_train,y_train)
y_pred = classifierCV.predict(X_test)
print(classifierCV.best_params_)
print("F1_Score:",f1_score(y_test,y_pred))

#### Tuning `min_samples_leaf`

In [ ]:
pipeline = Pipeline([('scalar',StandardScaler()),
                    ('dt',DecisionTreeClassifier(class_weight="balanced"))])
param_grid = {"dt__min_samples_leaf":[10,20,30,40,50,60,70]}

classifierCV=GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)

classifierCV.fit(X_train,y_train)
y_pred = classifierCV.predict(X_test)
print(classifierCV.best_params_)
print("F1_score: ",f1_score(y_test,y_pred))


### 6.1 Best Pre-Pruned Model

Applying the best parameters found: `max_depth=2`, `min_samples_leaf=40`.

In [ ]:
model = DecisionTreeClassifier(max_depth=2,min_samples_leaf=40,class_weight="balanced")
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("F1_Score:",f1_score(y_test,y_pred))

plt.figure(figsize=(12,10))
plot_tree(
    model,
    feature_names = X.columns,
    class_names = ["No","Yes"],
    filled=True
)
plt.tight_layout()
plt.show()

## 7. Post-Pruning — Cost-Complexity Pruning (CCP)

Post-pruning trims an already-trained full tree. The key parameter `ccp_alpha` controls how aggressively branches are removed.

- A **higher alpha** removes more branches (simpler tree, less overfitting)
- A **lower alpha** keeps more branches (complex tree, more overfitting)

We compute the pruning path and use `GridSearchCV` to find the optimal alpha.

In [ ]:
full_tree = DecisionTreeClassifier(class_weight="balanced")
full_tree.fit(X_train,y_train)

path = full_tree.cost_complexity_pruning_path(X_train,y_train)
ccp_alphas = path.ccp_alphas

pipeline = Pipeline([('scalar',StandardScaler()),
                     ('dt',DecisionTreeClassifier(class_weight="balanced"))])

param_grid = {"dt__ccp_alpha":ccp_alphas}
classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)
classifierCV.fit(X_train,y_train)
y_pred = classifierCV.predict(X_test)
print("Best params:",classifierCV.best_params_)
print("F1 Score:",f1_score(y_test,y_pred))

### 7.1 Best Post-Pruned Model

Applying the best `ccp_alpha` found from grid search.

In [ ]:
model = DecisionTreeClassifier(ccp_alpha=0.017785110000324428,class_weight="balanced")
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
print("F1 Score",f1_score(y_test,y_pred))

plt.figure(figsize=(12,10))
plot_tree(
    model,
    feature_names = X.columns,
    class_names = ["No","Yes"],
    filled=True
)
plt.tight_layout()
plt.show()

## 8. Final Results & Summary

| Model | F1-Score |
|---|---|
| Baseline (Unpruned) | ~0.57 |
| Pre-Pruned (`max_depth=2`) | 0.676 |
| Post-Pruned (CCP) | 0.676 |

**Key Findings:**
- Both pruning techniques significantly improve F1-Score over the baseline.
- Pre-pruning and post-pruning reach similar performance (~67.6%), suggesting the optimal tree complexity has been found.
- The model successfully exceeds the 55% benchmark target.

**Visualisations below:** High-resolution tree diagram and performance comparison chart.